# Grad Thesis Analysis
Google Earth Engine data retrieval and export to Dropbox.

In [ ]:
# Install dependencies
!pip install dropbox earthengine-api -q

In [ ]:
import time
import os
import dropbox
import pandas as pd
import io
import ee
from google.colab import drive, userdata

# Authenticate and initialize Google Earth Engine
ee.Authenticate()
ee.Initialize()

# Mount Google Drive
drive.mount('/content/drive')

# Connect to Dropbox (store your token in Colab Secrets as DROPBOX_TOKEN)
ACCESS_TOKEN = userdata.get('DROPBOX_TOKEN')
dbx = dropbox.Dropbox(ACCESS_TOKEN)
print("✓ GEE and Dropbox ready")

In [ ]:
def process_city_year(city, year, months=[5, 6, 7], n_samples=5000):
    """
    Compute mean NDVI and LST for a city over specified months of a given year.
    Uses Landsat 8 Collection 2 Level 2 (30m resolution).
    Returns an ee.FeatureCollection of sampled points with NDVI, LST, lat/lon.

    Args:
        city     : City name matching US Census TIGER/2018/Places (e.g. 'Corpus Christi')
        year     : Year as int (e.g. 2020)
        months   : List of month numbers to include (default: [5, 6, 7] = May/Jun/Jul)
        n_samples: Number of random sample points within city boundary
    """
    # --- City boundary from US Census TIGER dataset ---
    cities = ee.FeatureCollection('TIGER/2018/Places')
    city_boundary = cities.filter(ee.Filter.eq('NAME', city)).first().geometry()

    # --- Build merged ImageCollection for requested months only ---
    collections = []
    for month in months:
        start = ee.Date.fromYMD(year, month, 1)
        end   = start.advance(1, 'month')
        monthly = (
            ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .filterDate(start, end)
            .filterBounds(city_boundary)
            .filter(ee.Filter.lt('CLOUD_COVER', 20))
        )
        collections.append(monthly)

    merged = collections[0]
    for col in collections[1:]:
        merged = merged.merge(col)

    # --- Compute NDVI and LST per image ---
    def compute_ndvi_lst(image):
        # Landsat Collection 2 L2 scale factors
        optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
        thermal = image.select('ST_B10').multiply(0.00341802).add(149.0)

        # NDVI: (NIR - Red) / (NIR + Red) using B5=NIR, B4=Red
        ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

        # LST in Celsius (ST_B10 is in Kelvin)
        lst = thermal.subtract(273.15).rename('LST')

        return (image
                .addBands([ndvi, lst])
                .select(['NDVI', 'LST'])
                .set('month', image.date().get('month'))
                .set('year',  image.date().get('year')))

    processed = merged.map(compute_ndvi_lst)

    # --- Mean composite over all included images ---
    mean_image = processed.mean()

    # --- Random sample within city boundary ---
    samples = mean_image.sample(
        region=city_boundary,
        scale=30,
        numPixels=n_samples,
        seed=42,
        geometries=True
    )

    # --- Attach coordinates and metadata to each point ---
    def add_metadata(feature):
        coords = feature.geometry().coordinates()
        return feature.set({
            'longitude': coords.get(0),
            'latitude':  coords.get(1),
            'city':      city,
            'year':      year,
            'months':    '_'.join(str(m) for m in months),
        })

    return samples.map(add_metadata)

print("✓ process_city_year defined")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
city   = 'Corpus Christi'
year   = 2020
months = [5, 6, 7]   # May, June, July only

drive_folder   = 'GEE_Exports_final2'
dropbox_folder = '/GEE_Exports'
filename       = f'ndvi_lst_corpus_christi_{year}_may_jun_jul_sampled.csv'

drive_dir  = f'/content/drive/MyDrive/{drive_folder}'
drive_path = os.path.join(drive_dir, filename)

# ── Step 1: Start GEE export to Google Drive ─────────────────────────────────
print("Step 1: Starting GEE export...")
try:
    fc = process_city_year(city, year, months=months)
    export_task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f'NDVI_LST_{city.replace(" ", "_")}_{year}_May_Jun_Jul',
        folder=drive_folder,
        fileNamePrefix=filename.replace('.csv', ''),
        fileFormat='CSV'
    )
    export_task.start()
    print(f"✓ Export started  |  Task ID: {export_task.id}")
except Exception as e:
    raise RuntimeError(f"Failed to start GEE export: {e}")

# ── Step 2: Poll until GEE export is done ────────────────────────────────────
print("\nStep 2: Waiting for GEE export to finish...")
while True:
    state = export_task.status()['state']
    print(f"  Status: {state}")
    if state == 'COMPLETED':
        print("✓ Export complete")
        break
    elif state in ('FAILED', 'CANCELLED'):
        raise RuntimeError(f"GEE export ended with state: {state}")
    time.sleep(30)

# ── Step 3: Upload from Drive to Dropbox ─────────────────────────────────────
print("\nStep 3: Uploading to Dropbox...")
if not os.path.exists(drive_path):
    # Show what GEE actually wrote so you can spot filename mismatches
    actual = os.listdir(drive_dir) if os.path.isdir(drive_dir) else []
    raise FileNotFoundError(
        f"Expected file not found: {drive_path}\n"
        f"Files in Drive folder: {actual}"
    )

try:
    with open(drive_path, 'rb') as f:
        dbx.files_upload(
            f.read(),
            f'{dropbox_folder}/{filename}',
            mode=dropbox.files.WriteMode('overwrite')
        )
    print(f"✓ Uploaded to Dropbox: {dropbox_folder}/{filename}")
except Exception as e:
    raise RuntimeError(f"Dropbox upload failed: {e}")

# ── Step 4: Delete file from Google Drive ────────────────────────────────────
print("\nStep 4: Deleting file from Google Drive...")
os.remove(drive_path)
print(f"✓ Deleted from Drive: {drive_path}")

print("\n✓ All done — Drive is clear, data is in Dropbox.")

# To change the city, year, or months — edit the Configuration block in the cell above.
# months=[5, 6, 7] → May, June, July
# months=[1, 2, 3] → January, February, March
# etc.